In [36]:
import requests
import pandas as pd
import json

In [48]:
# parameters
start_date = '2025-09-25 00:00:00'
end_date = '2025-10-01 00:00:00'
tickers_list = ['BTC_USDT']
gate_url = 'https://api.gateio.ws/api/v4'

start_date_unix = int(pd.to_datetime(start_date).timestamp())
end_date_unix = int(pd.to_datetime(end_date).timestamp())

In [49]:
def make_a_request(url, params):
    
    request = requests.get(url, params)
    
    return request

In [50]:
def get_gate_historical_trades(ticker, start_time, end_time, page):
    
    url = gate_url + '/spot/trades'
    params = {
        'currency_pair': ticker,
        'from': start_time,
        'to': end_time,
        'limit': 1000,
        'page': 1
    }
    
    request = make_a_request(url, params)
    
    return request

In [51]:
def get_gate_date(ticker, start_time, end_time):
    
    df = pd.DataFrame()
    max_start_time = end_time
    while True:
        response = get_gate_historical_trades(ticker, start_time, max_start_time, page)
        response_json = response.json()
        response_df = pd.DataFrame(response_json)
        max_start_time = int(response_df['create_time'].min())
        df = pd.concat([df, response_df])
        if len(response_df) < 1000:
            break
    
    df = df.rename(columns={'amount': 'quantity', 'create_time_ms': 'unix_timestamp', 'side': 'taker_side', 'currency_pair': 'ticker'})
    df['price'] = df['price'].apply(lambda x: float(x))
    df['quantity'] = df['quantity'].apply(lambda x: float(x))
    df['unix_timestamp'] = df['unix_timestamp'].apply(lambda x: float(x))
    
    df['quantity_usdt'] = df['price']*df['quantity']
    df['timestamp'] = df['unix_timestamp'].apply(lambda x: pd.to_datetime(x, unit='ms'))
    
    df = df.drop_duplicates().sort_values(by='timestamp').reset_index(drop=True)
    df = df[['timestamp', 'price', 'quantity', 'taker_side', 'quantity_usdt']] # , 'unix_timestamp', 'ticker'
        
    return df

In [52]:
df = get_gate_date(tickers_list[0], start_date_unix, end_date_unix)

In [53]:
df

,timestamp,price,quantity,taker_side,quantity_usdt
0,2025-09-25 00:00:00.215397949,113302.4,0.000008,buy,0.906419
1,2025-09-25 00:00:00.265584961,113302.4,0.000013,buy,1.472931
2,2025-09-25 00:00:00.284552002,113302.4,0.000088,buy,9.970611
3,2025-09-25 00:00:00.351074951,113302.4,0.000132,buy,14.955917
4,2025-09-25 00:00:00.450241943,113302.4,0.000044,buy,4.985306
...,...,...,...,...,...
1382990,2025-09-30 23:59:41.735553955,114036.9,0.000001,sell,0.114037
1382991,2025-09-30 23:59:41.735579102,114036.9,0.000017,sell,1.938627
1382992,2025-09-30 23:59:50.447635986,114036.9,0.001407,sell,160.449918
1382993,2025-09-30 23:59:51.462810059,114037.0,0.000438,buy,49.948206


In [54]:
df.to_pickle("./raw_data_gate_BTCUSDT.pkl") 